# Projet Student
### Cours "Visualisation de Données"

---

> **🎯 Objectifs du projet**
> - Utiliser les différents jeux de données pour répondre à différentes questions
> - Faire des appels HTTP avec `requests` et visualiser les résultats avec Plotly
> - Encapsuler une API REST comme tool d'un agent LangGraph
> - Créer un serveur MCP minimal avec **FastMCP** et le connecter à un agent
> - Construire un pipeline complet : langage naturel → agent → API/MCP → réponse structurée

---

## Pip install requis pour lancer le projet :

```bash
pip install langchain langgraph langchain-openai langchain-mcp-adapters langchain-community langchain-ollama mcp requests duckdb streamlit nest-asyncio pandas plotly
```

### Imports des différentes bibliothèques

In [55]:
# Imports communs
import pandas as pd
import plotly.express as px
import plotly.io as pio
import webbrowser, json, re, os
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)  # LangGraph v1.0 migration warnings

from langchain_core.tools import tool
from langchain_openai import ChatOpenAI          # meilleure compatibilité tool calling
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient

pio.renderers.default = 'browser'
print("✓ Imports OK")

# ─── SETUP — Installation et imports ───────────────────────────────────────
import subprocess, sys

# Installer langchain-mcp-adapters si absent
# Paquets requis
for pkg, import_name in [
    ('langchain-mcp-adapters', 'langchain_mcp_adapters'),
    ('langchain-openai',       'langchain_openai'),
]:
    try:
        __import__(import_name)
        print(f'✓ {pkg} déjà installé')
    except ImportError:
        print(f'Installation de {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        print(f'✓ {pkg} installé')

# Imports principaux

warnings.filterwarnings('ignore', message='.*LangGraphDeprecated.*')


import os, json, webbrowser, asyncio
import nest_asyncio; nest_asyncio.apply()


from langchain_core.tools import tool

print('✓ Tous les imports OK')
print(f'Répertoire de travail : {os.getcwd()}')



✓ Imports OK
✓ langchain-mcp-adapters déjà installé
✓ langchain-openai déjà installé
✓ Tous les imports OK
Répertoire de travail : c:\UQAC\Eté\Séminaire IA\Projet Student


In [50]:
# ─── SETUP — Installation et imports ───────────────────────────────────────
import subprocess, sys

# Installer langchain-mcp-adapters si absent
# Paquets requis
for pkg, import_name in [
    ('langchain-mcp-adapters', 'langchain_mcp_adapters'),
    ('langchain-openai',       'langchain_openai'),
]:
    try:
        __import__(import_name)
        print(f'✓ {pkg} déjà installé')
    except ImportError:
        print(f'Installation de {pkg}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '-q'], check=True)
        print(f'✓ {pkg} installé')

# Imports principaux
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', message='.*LangGraphDeprecated.*')
import requests
import pandas as pd
import plotly.express as px
import os, json, webbrowser, asyncio
import nest_asyncio; nest_asyncio.apply()

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool

print('✓ Tous les imports OK')
print(f'Répertoire de travail : {os.getcwd()}')



✓ langchain-mcp-adapters déjà installé
✓ langchain-openai déjà installé
✓ Tous les imports OK
Répertoire de travail : c:\UQAC\Eté\Séminaire IA\Projet Student


### Définition et test des fonctions de base pour le traitement de datases

In [32]:


import re, glob
import pandas as pd

# ── Helper partagé : fuzzy matching robuste ──────────────────────────────────
def fuzzy_col(df, col_name: str) -> str:
    """Trouve la vraie colonne dans df à partir d'un nom approximatif."""
    col_clean = re.sub(r'[^a-zA-Z0-9 ]', '', col_name).strip().lower()
    for c in df.columns:
        if c.lower() == col_clean:
            return c
    for c in df.columns:
        if col_clean in c.lower() or c.lower() in col_clean:
            return c
    words = [w for w in col_clean.split() if len(w) >= 4]
    for word in words:
        stems = [word,
                 word[:-1] if len(word) > 4 else '',
                 word[:-2] if len(word) > 5 else '']
        for c in df.columns:
            if any(s and s in c.lower() for s in stems):
                return c
    return col_name

def _col_not_found_msg(df, col_name, kind='catégorielle'):
    if kind == 'catégorielle':
        available = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    else:
        available = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return (f"Colonne '{col_name}' introuvable. "
            f"Utilise un de ces noms exacts : {available}")

def _clean_filepath(filepath: str) -> str:
    """Sanitise le filepath : JSON array, faux chemin absolu, typo de nom."""
    fp = filepath.strip()
    # Cas 1 : tableau JSON  ["file.csv"]
    if fp.startswith("["):
        fp = fp.strip("[]").split(",")[0].strip()
        fp = fp.strip('"').strip("'")
    # Cas 2 : faux chemin absolu inventé par le LLM (/path/to/file.csv)
    if not os.path.exists(fp) and "/" in fp:
        basename = fp.split("/")[-1]
        if os.path.exists(basename):
            fp = basename
    # Cas 3 : typo dans le nom (job_posting.csv → job_postings.csv)
    if not os.path.exists(fp):
        name = os.path.basename(fp)
        candidates = glob.glob("*.csv")
        # prefer exact prefix match
        for c in candidates:
            if c.startswith(name.split(".")[0][:8]):
                fp = c
                break
        else:
            if len(candidates) == 1:
                fp = candidates[0]
    return fp

@tool
def describe_dataset(filepath: str = "ai_student_impact_dataset.csv") -> str:
    """Charge un CSV et retourne ses dimensions et colonnes. Utiliser EN PREMIER.
    Args:
        filepath: Chemin vers le fichier CSV à analyser.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'ai_student_impact_dataset.csv'."
    cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return (f"{df.shape[0]} lignes, {df.shape[1]} colonnes.\n"
            f"Catégorielles (utilise bar_chart, noms exacts) : {cat_cols}\n"
            f"Numériques (utilise numeric_stats, noms exacts) : {num_cols}")

@tool
def bar_chart(filepath: str = "ai_student_impact_dataset.csv", column: str = "", title: str = '') -> str:
    """Génère un bar chart horizontal des 10 valeurs les plus fréquentes. Sauvegarde en HTML.
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne catégorielle.
        title: Titre du graphique (optionnel).
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'ai_student_impact_dataset.csv'."
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
    top = df[col].value_counts().head(10).reset_index()
    top.columns = [col, 'count']
    fig = px.bar(top.sort_values('count'), x='count', y=col, orientation='h', title=title or col)
    out = f'chart_{col.replace(" ", "_")}.html'
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Graphique sauvegardé : {out}"



# ── Tests directs ─────────────────────────────────────────────────────────────
print("=== describe_dataset ===")
print("--- Impact de l'IA ---")
print(describe_dataset.invoke({"filepath": "ai_student_impact_dataset.csv"}))
print("\n=== bar_chart ===")
print(bar_chart.invoke({"filepath": "ai_student_impact_dataset.csv",
                         "column": "Prompt_Engineering_Skill",
                         "title": "Prompt skills"}))
print("--- Réseaux sociaux et santé mentale ---")
print(describe_dataset.invoke({"filepath": "Student Social Media And Mental Health Impact.csv"}))
print("--- Performences des étudiants ---")
print(describe_dataset.invoke({"filepath": "StudentPerformanceFactors.csv"}))


=== describe_dataset ===
--- Impact de l'IA ---
50000 lignes, 16 colonnes.
Catégorielles (utilise bar_chart, noms exacts) : ['Major_Category', 'Year_of_Study', 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Institutional_Policy', 'Burnout_Risk_Level']
Numériques (utilise numeric_stats, noms exacts) : ['Student_ID', 'Pre_Semester_GPA', 'Weekly_GenAI_Hours', 'Tool_Diversity', 'Paid_Subscription', 'Traditional_Study_Hours', 'Perceived_AI_Dependency', 'Anxiety_Level_During_Exams', 'Post_Semester_GPA', 'Skill_Retention_Score']

=== bar_chart ===
Graphique sauvegardé : chart_Prompt_Engineering_Skill.html
--- Réseaux sociaux et santé mentale ---
5000 lignes, 13 colonnes.
Catégorielles (utilise bar_chart, noms exacts) : ['Gender', 'Country', 'Academic_Level', 'Most_Used_Platform', 'Purpose_Of_Use', 'Stress_Level']
Numériques (utilise numeric_stats, noms exacts) : ['Age', 'Avg_Daily_Usage_Hours', 'Daily_Unlocks', 'Study_Hours', 'Physical_Activity_Hours', 'Sleep_Hours_Per_Night', 'Mental_Healt

## Définition d'autres fonctions d'affichage :

> **Un graph de points**

> **Un affichage cammenbert**

In [170]:
#Autres fonctions pour des graphiques
import webbrowser

@tool
def scatter_correlation(filepath: str, x_col: str, y_col: str, color_col: str = None,title: str = '') -> str:
    """Génère un graphique de corrélation (nuage de points) entre deux colonnes numériques. Sauvegarde en HTML.
    Args:
        filepath: Chemin vers le fichier CSV.
        x_col: Nom de la colonne numérique pour l'axe X.
        y_col: Nom de la colonne numérique pour l'axe Y.
        color_col: Colonne optionnelle pour colorer les points (catégorielle).
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable."

    x = fuzzy_col(df, x_col)
    y = fuzzy_col(df, y_col)
    c = fuzzy_col(df, color_col) if color_col else None
    
    if x not in df.columns or y not in df.columns:
        return f"Colonnes introuvables pour le scatter plot. X: '{x}', Y: '{y}'."

    fig = px.scatter(df, x=x, y=y, color=c, title=f"Corrélation : {x} vs {y}")
    
    # Sécuriser le nom du fichier HTML de sortie
    filename_x = x.replace(" ", "_").replace("/", "_")
    filename_y = y.replace(" ", "_").replace("/", "_")
    out = f'scatter_{filename_x}_vs_{filename_y}.html'
    
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Graphique de corrélation sauvegardé et ouvert : {out}"


@tool
def pie_distribution(filepath: str, column: str, title: str ='') -> str:
    """Génère un graphique en camembert (Pie chart) pour analyser la répartition (top 5). Sauvegarde en HTML.
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom de la colonne catégorielle à analyser.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable."
        
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
    
    top = df[col].value_counts().head(5).reset_index()
    top.columns = [col, 'count']
    
    fig = px.pie(top, values='count', names=col, title=f"Répartition (Top 5) de {col}")
    
    # Sécuriser le nom du fichier HTML de sortie
    out = f'pie_{col.replace(" ", "_").replace("/", "_")}.html'
    
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Graphique en camembert sauvegardé et ouvert : {out}"

> **Test de ces nouvelles fonctions**

In [171]:
#Test de ces nouvelles fonctions
print("\n=== scatter_correlation ===")
print(scatter_correlation.invoke({"filepath": "ai_student_impact_dataset.csv",
                         "x_col": "Weekly_GenAI_Hours",
                         "y_col": "Post_Semester_GPA",
                         "title": "Weekly impact"}))

print("\n=== Pie distribution ===")
print(pie_distribution.invoke({"filepath": "ai_student_impact_dataset.csv",
                         "column": "Major_Category",
                         "title": "Major_Category"}))


=== scatter_correlation ===
Graphique de corrélation sauvegardé et ouvert : scatter_Weekly_GenAI_Hours_vs_Post_Semester_GPA.html

=== Pie distribution ===
Graphique en camembert sauvegardé et ouvert : pie_Major_Category.html


## Définition des tools de traitement des colones

>**Pour colones catégorielles**

In [33]:
@tool
def top_values(filepath: str = "ai_student_impact_dataset.csv", column: str = "") -> str:
    """Retourne les 10 valeurs (s'il y en a 10) les plus fréquentes d'une colonne catégorielle (texte).
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne catégorielle.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'ai_student_impact_dataset.csv'."
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
    result = df[col].value_counts()
    n_items = len(result)
    if n_items>10:
        result=result.head(10)
        titre = f"Top 10 — {col} :\n"
    else:
        titre = f"Top {n_items} — {col} :\n"
    return titre + result.to_string()

print(top_values.invoke({"filepath": "ai_student_impact_dataset.csv", "column": "Primary_Use_Case"}))

Top 5 — Primary_Use_Case :
Primary_Use_Case
Debugging/Troubleshooting    12295
Copywriting/Drafting         12011
Ideation                     10721
Summarizing_Reading           8633
Direct_Answer_Generation      6340


>**Pour colones numériques**

In [34]:
@tool
def numeric_stats(filepath: str, column: str) -> str:
    """TODO : Retourne min, max, moyenne, médiane d'une colonne numérique
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne numérique.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'ai_student_impact_dataset.csv'."
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'numérique')
    serie = df[col].dropna()

    return (
    f"Statistiques — {col} :\n"
    f"Moyenne : {serie.mean():.2f}\n"
    f"Médiane : {serie.median():.2f}\n"
    f"Minimum : {serie.min():.2f}\n"
    f"Maximum : {serie.max():.2f}"
)
    pass

# Test
print(numeric_stats.invoke({"filepath": "ai_student_impact_dataset.csv", "column": "Anxiety_Level_During_Exams"}))

Statistiques — Anxiety_Level_During_Exams :
Moyenne : 4.27
Médiane : 4.00
Minimum : 1.00
Maximum : 10.00


**Définition du show_step**

In [35]:

# Mots-clés qui indiquent qu'un tool a sauvegardé un fichier.
# Si le tool a réussi, on affiche son résultat plutôt que la réponse du LLM
# (llama3.2 hallucine souvent des tableaux/stats après un tool qui génère un fichier).
_SAVED_KEYWORDS = ("sauvegardé", "saved", "Dashboard sauvegardé", "Graphique sauvegardé")

def show_steps(agent, question: str, config: dict = None):
    """Lance l'agent et affiche chaque etape en temps reel."""
    print(f"► Question : {question}")
    print("─" * 60)

    final = ""
    last_tool_result = ""   # dernier resultat de tool (utilise si le LLM hallucine)

    for chunk in agent.stream({"messages": [("human", question)]}, config or {}):

        if "agent" in chunk:
            msg = chunk["agent"]["messages"][0]
            if hasattr(msg, 'tool_calls') and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  ▶ Appel tool : {tc['name']}")
                    print(f"    Arguments  : {tc['args']}")
            else:
                final = msg.content

        elif "tools" in chunk:
            obs = chunk["tools"]["messages"][0].content
            print(f"  ◀ Résultat  : {obs[:150]}")
            if any(kw in obs for kw in _SAVED_KEYWORDS):
                last_tool_result = obs.split("\n")[0]

    print("─" * 60)

    # Si le tool a sauvegarde un fichier, ignorer la reponse du LLM
    # (evite les tableaux/stats hallucines que llama3.2 genere apres make_dashboard/bar_chart)
    if last_tool_result and any(kw in last_tool_result for kw in _SAVED_KEYWORDS):
        print(f"✓ {last_tool_result}")
    else:
        print(f"✓ Réponse finale : {final[:200]}")

    return final

print("✓ show_steps() prêt")


✓ show_steps() prêt


**Définition  et test du make_dashboard**

In [36]:
from plotly.subplots import make_subplots
import plotly.graph_objects as go

@tool
def make_dashboard(filepath: str, column1: str, column2: str, title: str) -> str:
    """Génère un dashboard HTML avec 2 bar charts côte à côte. Sauvegarde et ouvre dans le navigateur.
    IMPORTANT : utilise uniquement des noms de colonnes retournés par describe_dataset.
    Args:
        filepath: Chemin vers le fichier CSV.
        column1: Première colonne catégorielle (nom exact ou partiel).
        column2: Deuxième colonne catégorielle (nom exact ou partiel).
        title: Titre du dashboard.
    """
    df = pd.read_csv(_clean_filepath(filepath))

    # Normaliser et vérifier les deux colonnes
    col1 = fuzzy_col(df, column1)
    col2 = fuzzy_col(df, column2)

    # Si une colonne est introuvable → retourner la liste des colonnes disponibles
    # (le LLM lira ce message et relancera avec les bons noms)
    if col1 not in df.columns or col2 not in df.columns:
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        missing = [c for c, real in [(column1, col1), (column2, col2)] if real not in df.columns]
        return (f"Colonnes introuvables : {missing}. "
                f"Utilise ces noms exacts : {cat_cols}")

    # Préparer les données : top 10 pour chaque colonne
    def top10(col):
        t = df[col].value_counts().head(10).reset_index()
        t.columns = [col, 'count']
        return t.sort_values('count')

    # Créer la figure avec 2 sous-graphiques côte à côte
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[col1, col2],
                        horizontal_spacing=0.15)
    t1 = top10(col1)
    fig.add_trace(go.Bar(x=t1['count'], y=t1[col1], orientation='h', name=col1), row=1, col=1)
    t2 = top10(col2)
    fig.add_trace(go.Bar(x=t2['count'], y=t2[col2], orientation='h', name=col2), row=1, col=2)
    fig.update_layout(title_text=title, height=500, showlegend=False)

    out = 'dashboard.html'
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Dashboard sauvegardé : {out} ({col1} + {col2})"

# Test direct
print(make_dashboard.invoke({
    "filepath": "ai_student_impact_dataset.csv",
    "column1": "Major_Category",
    "column2": "Primary_Use_Case",
    "title": "Analyse de l'utilisation de l'IA"
}))

Dashboard sauvegardé : dashboard.html (Major_Category + Primary_Use_Case)


# Création d'un premier agent

## Définition de l'agent avec mémoire

In [ ]:

# ── 1. Créer le LLM ──────────────────────────────────────────────────────────
# ChatOpenAI pointe sur l'API Ollama en mode compatibilité OpenAI (port 11434/v1)
# Plus fiable que ChatOllama pour le tool calling avec llama3.2
llm = ChatOpenAI(
    model="llama3.2",
    base_url="http://localhost:11434/v1",
    api_key="ollama" ,              # clé fictive requise par l'API
    temperature=0
)

memory = MemorySaver()

config = {"configurable": {"thread_id": "projet_student"}}

tools = [describe_dataset, bar_chart, top_values, numeric_stats,make_dashboard,scatter_correlation,pie_distribution]

# ── 2. Créer l'agent ─────────────────────────────────────────────────────────
# System prompt : évite que llama3.2 hallucine du HTML après l'exécution du tool.
# Règle universelle pour ce TP : quand un tool confirme qu'un fichier est sauvegardé,
# répondre avec une seule phrase — jamais de HTML, SVG ou lien <a href>.
PROMPT = (
    "You are a data visualization assistant working with ai_student_impact_dataset.csv. "
    "CRITICAL RULES: "
    "1. Always call tools directly — never describe a tool call as text. "
    "2. Use EXACT column names as given by the user — NEVER abbreviate or shorten them. "
    "   Example: use 'Company Industry', NOT 'Company'. Use 'Job Position Type', NOT 'Job Type'. "
    "3. After any tool call, your ENTIRE response must be exactly: "
    "   'Dashboard generated: dashboard.html' — nothing else. "
    "4. Do NOT write tables, lists, statistics, HTML, or any description. Stop immediately."
)
agent = create_react_agent(llm, tools,checkpointer=memory, prompt=PROMPT)

# ── 3. Lancer en langage naturel avec plusieurs demandes ─────────────────────────────────────────────
show_steps(
    agent,
    "Explore ai_student_impact_dataset.csv, calcul les statistiques de 'Weekly_GenAI_Hours' " 
    "ET génère un bar chart pour la colonne 'Major_Category'.",
    config
)


► Question : Explore ai_student_impact_dataset.csv, calcul les statistiques de 'Weekly_GenAI_Hours' ET génère un bar chart pour la colonne 'Major_Category'.
────────────────────────────────────────────────────────────
  ▶ Appel tool : numeric_stats
    Arguments  : {'filepath': 'ai_student_impact_dataset.csv', 'column': 'Weekly_GenAI_Hours'}
  ▶ Appel tool : bar_chart
    Arguments  : {'filepath': 'ai_student_impact_dataset.csv', 'column': 'Major_Category'}
  ◀ Résultat  : Statistiques — Weekly_GenAI_Hours :
Moyenne : 8.43
Médiane : 5.80
Minimum : 0.00
Maximum : 40.00
  ◀ Résultat  : Graphique sauvegardé : chart_Major_Category.html
────────────────────────────────────────────────────────────
✓ Graphique sauvegardé : chart_Major_Category.html


'Dashboard generated: chart_Major_Category.html'

## Tests de l'agent

**Test du système de mémoire**

In [ ]:

# ── 4. Test de la mémoire ─────────────────────────────────────────────
# ── Question 1 : exploration ──────────────────────────────────────────────────
# On précise l'outil à utiliser pour éviter que llama3.2 appelle top_values à la place
print("=== Question 1 : exploration ===")
show_steps(
    agent,
    "Utilise describe_dataset sur ai_student_impact_dataset.csv et liste les colonnes catégorielles.",
    config
)

# ── Question 2 : graphique basé sur Q1 ───────────────────────────────────────
# L'agent connaît les colonnes depuis Q1 — on cite une colonne catégorielle valide
print("\n=== Question 2 : graphique basé sur la Q1 ===")
show_steps(
    agent,
    "Génère un bar chart pour la colonne 'Primary_Use_Case' que tu viens de voir.",
    config
)

=== Question 1 : exploration ===
► Question : Utilise describe_dataset sur ai_student_impact_dataset.csv et liste les colonnes catégorielles.
────────────────────────────────────────────────────────────
  ▶ Appel tool : describe_dataset
    Arguments  : {'filepath': 'ai_student_impact_dataset.csv'}
  ◀ Résultat  : 50000 lignes, 16 colonnes.
Catégorielles (utilise bar_chart, noms exacts) : ['Major_Category', 'Year_of_Study', 'Primary_Use_Case', 'Prompt_Engineerin
────────────────────────────────────────────────────────────
✓ Réponse finale : Dashboard generated: dashboard.html

=== Question 2 : graphique basé sur la Q1 ===
► Question : Génère un bar chart pour la colonne 'Primary_Use_Case' que tu viens de voir.
────────────────────────────────────────────────────────────
  ▶ Appel tool : bar_chart
    Arguments  : {'column': 'Primary_Use_Case', 'filepath': 'ai_student_impact_dataset.csv'}
  ◀ Résultat  : Graphique sauvegardé : chart_Primary_Use_Case.html
────────────────────────────────

'Dashboard generated: chart_Primary_Use_Case.html'

**Test de l'utilisation de make_dashboard**

In [39]:

# Les noms de colonnes sont cités explicitement pour guider llama3.2
show_steps(
    agent,
    "Appelle make_dashboard avec column1='Major_Category', column2='Primary_Use_Case', "
    "filepath='ai_student_impact_dataset.csv', title='Analyse utilisation IA'.",
    config
)

► Question : Appelle make_dashboard avec column1='Major_Category', column2='Primary_Use_Case', filepath='ai_student_impact_dataset.csv', title='Analyse utilisation IA'.
────────────────────────────────────────────────────────────
  ▶ Appel tool : make_dashboard
    Arguments  : {'column1': 'Major_Category', 'column2': 'Primary_Use_Case', 'filepath': 'ai_student_impact_dataset.csv', 'title': 'Analyse utilisation IA'}
  ◀ Résultat  : Dashboard sauvegardé : dashboard.html (Major_Category + Primary_Use_Case)
────────────────────────────────────────────────────────────
✓ Dashboard sauvegardé : dashboard.html (Major_Category + Primary_Use_Case)


'Dashboard generated: dashboard.html (Major_Category + Primary_Use_Case)'

# Création des nouveaux tools pour les datasets

## Tools pour le dataset de l'impact de l'IA sur les étudiants

**Définition des tools**

In [40]:
# On créer les tools pour le dataset sur l'impact de l'IA.
@tool
def analyse_impact_ia_par_groupe(filepath: str,column: str = 'Primary_Use_Case') -> str:
    """
    Analyse l'impact de l'IA en regroupant par une colonne catégorielle spécifique.
    Colonnes valides recommandées : 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Major_Category'
    """
    colonnes_valides = ['Primary_Use_Case', 'Prompt_Engineering_Skill', 'Major_Category', 'Burnout_Risk_Level']
    if column not in colonnes_valides:
        return f"Erreur : La colonne de segmentation doit être l'une des suivantes : {colonnes_valides}"
    
    df_ia = pd.read_csv(_clean_filepath(filepath))
        
    # Groupement et calcul des moyennes clés
    analyse = df_ia.groupby(column).agg({
        'Pre_Semester_GPA': 'mean',
        'Post_Semester_GPA': 'mean',
        'Weekly_GenAI_Hours': 'mean',
        'Skill_Retention_Score': 'mean',
        'Anxiety_Level_During_Exams': 'mean'
    }).round(2).reset_index()
    
    # Ajout d'une colonne calculée pour voir l'évolution globale du GPA par groupe
    analyse["Evolution_GPA"] = (analyse["Post_Semester_GPA"] - analyse["Pre_Semester_GPA"]).round(2)
    
    rapport = f"### 🤖 ANALYSE DE L'IA SEGMENTÉE PAR : {column}\n\n"
    rapport += analyse.to_markdown(index=False)
    
    return rapport

@tool
def detecter_dependance_ia(filepath: str,seuil_heures: int = 15) -> str:
    """
    Filtre les étudiants ayant une utilisation intensive de l'IA (défini par le seuil d'heures)
    et compare leur profil à la moyenne globale de tous les étudiants.
    """
    df_ia = pd.read_csv(_clean_filepath(filepath))

    # Groupe "Intensif" vs Groupe "Global"
    df_intensif = df_ia[df_ia['Weekly_GenAI_Hours'] >= seuil_heures]
    
    metrics = [
        'Pre_Semester_GPA', 'Post_Semester_GPA', 'Weekly_GenAI_Hours', 
        'Traditional_Study_Hours', 'Anxiety_Level_During_Exams', 'Skill_Retention_Score'
    ]
    
    moyenne_intensif = df_intensif[metrics].mean().to_frame(name=f'Utilisateurs Intensifs (>= {seuil_heures}h)')
    moyenne_globale = df_ia[metrics].mean().to_frame(name='Moyenne Globale')
    
    # Fusion des deux analyses pour comparaison directe
    comparatif = moyenne_globale.join(moyenne_intensif).round(2).reset_index()
    comparatif.rename(columns={'index': 'Indicateurs'}, inplace=True)
    
    rapport = f"### ⚠️ PROFIL COMPARAISON : FORTE DÉPENDANCE À L'IA (Effectif : {len(df_intensif)} étudiants)\n\n"
    rapport += comparatif.to_markdown(index=False)
    
    return rapport


@tool
def impact_temps_utilisation(filepath: str) -> str:
    """
    Analyse l'impact du temps d'utilisation hebdomadaire de l'IA sur le succès académique 
    (GPA, rétention des compétences) et le bien-être (anxiété, burnout) des étudiants.
    """
    df_ia = pd.read_csv(_clean_filepath(filepath))
    df = df_ia.copy()
    
    # 1. Définition des tranches d'utilisation basées sur 'Weekly_GenAI_Hours'
    # Moins de 5h = Faible | Entre 5h et 15h = Modéré | Plus de 15h = Intensif
    def segmenter_heures(heures):
        if heures < 5:
            return "1. Faible (< 5h/semaine)"
        elif heures <= 15:
            return "2. Modéré (5h - 15h/semaine)"
        else:
            return "3. Intensif (> 15h/semaine)"
            
    df['Tranche_Utilisation'] = df['Weekly_GenAI_Hours'].apply(segmenter_heures)
    
    # 2. Agrégation des statistiques clés par tranche d'utilisation
    analyse_impact = df.groupby('Tranche_Utilisation').agg({
        'Student_ID': 'count',                 # Nombre d'étudiants dans le groupe
        'Pre_Semester_GPA': 'mean',
        'Post_Semester_GPA': 'mean',
        'Traditional_Study_Hours': 'mean',     # Permet de voir s'ils étudient moins "traditionnellement"
        'Skill_Retention_Score': 'mean',       # Impact sur l'apprentissage profond
        'Anxiety_Level_During_Exams': 'mean'   # Impact psychologique
    }).round(2).reset_index()
    
    # Renommer la colonne count pour plus de clarté
    analyse_impact.rename(columns={'Student_ID': 'Effectif_Etudiants'}, inplace=True)
    
    # 3. Calcul de l'évolution du GPA pour chaque groupe
    analyse_impact["Evolution_GPA"] = (analyse_impact["Post_Semester_GPA"] - analyse_impact["Pre_Semester_GPA"]).round(2)
    
    # Réorganisation des colonnes pour une lecture logique par l'agent
    colonnes_ordre = [
        'Tranche_Utilisation', 'Effectif_Etudiants', 'Traditional_Study_Hours',
        'Pre_Semester_GPA', 'Post_Semester_GPA', 'Evolution_GPA',
        'Skill_Retention_Score', 'Anxiety_Level_During_Exams'
    ]
    
    # Construction du rapport Markdown
    rapport = "### ⏱️ ANALYSE DE L'IMPACT DU TEMPS D'UTILISATION DE L'IA\n\n"
    rapport += "Ce tableau compare le profil des étudiants selon leur volume horaire hebdomadaire sur les outils d'IA :\n\n"
    rapport += analyse_impact[colonnes_ordre].to_markdown(index=False)
    
    return rapport

**Test des tools**

In [41]:
# Test des tools pour l'impact de l'ia

# Test analyse_impact
print(analyse_impact_ia_par_groupe.invoke({
    "filepath": "ai_student_impact_dataset.csv",
    "column": "Primary_Use_Case",
}))



### 🤖 ANALYSE DE L'IA SEGMENTÉE PAR : Primary_Use_Case

| Primary_Use_Case          |   Pre_Semester_GPA |   Post_Semester_GPA |   Weekly_GenAI_Hours |   Skill_Retention_Score |   Anxiety_Level_During_Exams |   Evolution_GPA |
|:--------------------------|-------------------:|--------------------:|---------------------:|------------------------:|-----------------------------:|----------------:|
| Copywriting/Drafting      |               3.13 |                3.34 |                 7.63 |                   75.23 |                         4.2  |            0.21 |
| Debugging/Troubleshooting |               3.15 |                3.4  |                 9.4  |                   78.07 |                         4.33 |            0.25 |
| Direct_Answer_Generation  |               3.16 |                3.29 |                 8.46 |                   73.72 |                         4.29 |            0.13 |
| Ideation                  |               3.14 |                3.34 |                 

In [42]:
# Test temps utilissation
print(impact_temps_utilisation.invoke({
    "filepath": "ai_student_impact_dataset.csv"
}))



### ⏱️ ANALYSE DE L'IMPACT DU TEMPS D'UTILISATION DE L'IA

Ce tableau compare le profil des étudiants selon leur volume horaire hebdomadaire sur les outils d'IA :

| Tranche_Utilisation          |   Effectif_Etudiants |   Traditional_Study_Hours |   Pre_Semester_GPA |   Post_Semester_GPA |   Evolution_GPA |   Skill_Retention_Score |   Anxiety_Level_During_Exams |
|:-----------------------------|---------------------:|--------------------------:|-------------------:|--------------------:|----------------:|------------------------:|-----------------------------:|
| 1. Faible (< 5h/semaine)     |                22567 |                     11.8  |               3.15 |                3.34 |            0.19 |                   75.97 |                         3.85 |
| 2. Modéré (5h - 15h/semaine) |                18899 |                     11.17 |               3.14 |                3.37 |            0.23 |                   77    |                         4.3  |
| 3. Intensif (> 15h/semaine

In [43]:
# Test dependance
print(detecter_dependance_ia.invoke({
    "filepath": "ai_student_impact_dataset.csv",
    "seuil_heures": 40
}))

### ⚠️ PROFIL COMPARAISON : FORTE DÉPENDANCE À L'IA (Effectif : 448 étudiants)

| Indicateurs                |   Moyenne Globale |   Utilisateurs Intensifs (>= 40h) |
|:---------------------------|------------------:|----------------------------------:|
| Pre_Semester_GPA           |              3.15 |                              3.14 |
| Post_Semester_GPA          |              3.35 |                              3.27 |
| Weekly_GenAI_Hours         |              8.43 |                             40    |
| Traditional_Study_Hours    |             11.21 |                              8.54 |
| Anxiety_Level_During_Exams |              4.27 |                              6.23 |
| Skill_Retention_Score      |             75.8  |                             64.66 |


On remarque que les tools fonctionnent bien comme on le souhaite.

## Tools pour le dataset des performances des étudiants

**Définition des tools**

In [44]:
# On créer les tools pour les performances des étudiants.

@tool
def top_progression(filepath : str,top_n: int = 15) -> str:
    """
    Calcule la progression des étudiants (Exam_Score - Previous_Scores),
    génère le top N des meilleures progressions et fournit la moyenne de leur profil.
    """

    df_perf = pd.read_csv(_clean_filepath(filepath))
    df = df_perf.copy()
    
    # Calcul de la variable de progression
    df['Progression'] = df['Exam_Score'] - df['Previous_Scores']
    
    # Extraction du Top N
    top_students = df.sort_values(by='Progression', ascending=False).head(top_n)
    
    # Sélection des colonnes numériques clés pour l'analyse
    cols_numeriques = ['Progression', 'Hours_Studied', 'Attendance', 'Sleep_Hours', 'Tutoring_Sessions']
    cols_cat = ['Motivation_Level', 'Teacher_Quality', 'Parental_Involvement']
    
    # 1. Calcul des moyennes du groupe d'élite
    moyennes = top_students[cols_numeriques].mean().to_frame().T
    moyennes.index = ['MOYENNE DU TOP']
    
    # 2. Préparation du tableau des étudiants
    affichage_students = top_students[cols_numeriques + cols_cat].copy()
    
    # Construction du rendu textuel final pour l'agent
    rapport = f"### 📈 TOP {top_n} DES MEILLEURES PROGRESSIONS APPRÉCIABLES\n\n"
    rapport += affichage_students.to_markdown(index=False)
    rapport += "\n\n### 📊 PROFIL STATISTIQUE MOYEN DE CE GROUPE\n\n"
    rapport += moyennes.to_markdown(index=False)
    
    return rapport

@tool
def analyse_par_type_ecole(filepath : str) -> str:
    """
    Compare les performances et l'accès aux ressources entre les écoles publiques et privées.
    """

    df_perf = pd.read_csv(_clean_filepath(filepath))

    # Agrégation des notes et variables critiques par School_Type
    stats_ecole = df_perf.groupby('School_Type').agg({
        'Exam_Score': 'mean',
        'Attendance': 'mean',
        'Hours_Studied': 'mean',
        'Previous_Scores': 'mean'
    }).round(2).reset_index()
    
    # Distribution des ressources en % (ex: Internet_Access)
    distribution_internet = pd.crosstab(df_perf['School_Type'], df_perf['Internet_Access'], normalize='index') * 100
    distribution_internet = distribution_internet.round(1).reset_index()
    
    rapport = "### 🏢 COMPARAISON PUBLIC VS PRIVÉ (MOYENNES)\n\n"
    rapport += stats_ecole.to_markdown(index=False)
    rapport += "\n\n### 🌐 ACCÈS À INTERNET PAR TYPE D'ÉCOLE (EN %)\n\n"
    rapport += distribution_internet.to_markdown(index=False)
    
    return rapport

@tool
def facteurs_echec_reussite(filepath : str) -> str:
    """
    Isole les 10% des meilleurs étudiants et les 10% des étudiants en difficulté 
    sur leur Exam_Score, puis compare leurs profils (habitudes d'étude, sommeil, environnement).
    """
    df_perf = pd.read_csv(_clean_filepath(filepath))
    df = df_perf.copy()
    
    # 1. Détermination des seuils pour les top 10% et bottom 10%
    seuil_haut = df['Exam_Score'].quantile(0.90)
    seuil_bas = df['Exam_Score'].quantile(0.10)
    
    # 2. Création des deux sous-groupes
    df_top = df[df['Exam_Score'] >= seuil_haut]
    df_bottom = df[df['Exam_Score'] <= seuil_bas]
    
    # 3. Métriques numériques clés à comparer
    metrics_num = [
        'Exam_Score', 'Hours_Studied', 'Attendance', 
        'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity'
    ]
    
    moyennes_top = df_top[metrics_num].mean().to_frame(name='Top 10% (Réussite)')
    moyennes_bottom = df_bottom[metrics_num].mean().to_frame(name='Bottom 10% (Difficulté)')
    
    # Fusion des analyses numériques
    comparatif_num = moyennes_top.join(moyennes_bottom).round(2).reset_index()
    comparatif_num.rename(columns={'index': 'Indicateurs Numériques'}, inplace=True)
    
    # 4. Métriques catégorielles clés (Facteurs environnementaux)
    # On va calculer le taux d'accès ou de forte implication pour donner du contexte social
    facteurs_env = []
    
    # % Accès Internet Élevé
    facteurs_env.append({
        'Facteur Environnemental': 'Accès Internet (Oui %)',
        'Top 10% (Réussite)': f"{(df_top['Internet_Access'] == 'Yes').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Internet_Access'] == 'Yes').mean() * 100:.1f}%"
    })
    
    # % Implication Parentale Forte
    facteurs_env.append({
        'Facteur Environnemental': 'Implication Parentale (High %)',
        'Top 10% (Réussite)': f"{(df_top['Parental_Involvement'] == 'High').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Parental_Involvement'] == 'High').mean() * 100:.1f}%"
    })
    
    # % Motivation Élevée
    facteurs_env.append({
        'Facteur Environnemental': 'Motivation de l\'élève (High %)',
        'Top 10% (Réussite)': f"{(df_top['Motivation_Level'] == 'High').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Motivation_Level'] == 'High').mean() * 100:.1f}%"
    })
    
    # % Qualité Enseignement Excellente
    facteurs_env.append({
        'Facteur Environnemental': 'Qualité Enseignant (High %)',
        'Top 10% (Réussite)': f"{(df_top['Teacher_Quality'] == 'High').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Teacher_Quality'] == 'High').mean() * 100:.1f}%"
    })
    
    comparatif_cat = pd.DataFrame(facteurs_env)
    
    # 5. Construction du rapport final
    rapport = f"### 🔑 ANALYSE DES FACTEURS DE RÉUSSITE VS ÉCHEC\n"
    rapport += f"Échantillon basé sur les extrêmes du jeu de données (Seuil Réussite >= {seuil_haut} pts | Seuil Difficulté <= {seuil_bas} pts).\n\n"
    
    rapport += "#### 📊 COMPARAISON DES HABITUDES (MOYENNES)\n\n"
    rapport += comparatif_num.to_markdown(index=False)
    
    rapport += "\n\n#### 🏡 COMPARAISON DE L'ENVIRONNEMENT ET CADRE SOCIAL\n\n"
    rapport += comparatif_cat.to_markdown(index=False)
    
    return rapport

**Test des tools**

In [45]:
# Test des tools pour les performances

# Test top_progression
print(top_progression.invoke({
    "filepath": "StudentPerformanceFactors.csv",
    "top_n": 10,
}))

### 📈 TOP 10 DES MEILLEURES PROGRESSIONS APPRÉCIABLES

|   Progression |   Hours_Studied |   Attendance |   Sleep_Hours |   Tutoring_Sessions | Motivation_Level   | Teacher_Quality   | Parental_Involvement   |
|--------------:|----------------:|-------------:|--------------:|--------------------:|:-------------------|:------------------|:-----------------------|
|            41 |              18 |           90 |             6 |                   1 | Low                | High              | High                   |
|            40 |              31 |           69 |             7 |                   2 | Medium             | Low               | Medium                 |
|            37 |              25 |           73 |             7 |                   2 | Medium             | Medium            | Medium                 |
|            35 |              19 |           70 |             7 |                   0 | High               | Medium            | Medium                 |
|            34

In [46]:
# Test type_école
print(analyse_par_type_ecole.invoke({
    "filepath": "StudentPerformanceFactors.csv",
}))

### 🏢 COMPARAISON PUBLIC VS PRIVÉ (MOYENNES)

| School_Type   |   Exam_Score |   Attendance |   Hours_Studied |   Previous_Scores |
|:--------------|-------------:|-------------:|----------------:|------------------:|
| Private       |        67.29 |        80.3  |           19.97 |             74.78 |
| Public        |        67.21 |        79.84 |           19.98 |             75.2  |

### 🌐 ACCÈS À INTERNET PAR TYPE D'ÉCOLE (EN %)

| School_Type   |   No |   Yes |
|:--------------|-----:|------:|
| Private       |  8.1 |  91.9 |
| Public        |  7.3 |  92.7 |


In [47]:
# Test facteur
print(facteurs_echec_reussite.invoke({
    "filepath": "StudentPerformanceFactors.csv"
}))

### 🔑 ANALYSE DES FACTEURS DE RÉUSSITE VS ÉCHEC
Échantillon basé sur les extrêmes du jeu de données (Seuil Réussite >= 72.0 pts | Seuil Difficulté <= 63.0 pts).

#### 📊 COMPARAISON DES HABITUDES (MOYENNES)

| Indicateurs Numériques   |   Top 10% (Réussite) |   Bottom 10% (Difficulté) |
|:-------------------------|---------------------:|--------------------------:|
| Exam_Score               |                74.19 |                     61.79 |
| Hours_Studied            |                25.53 |                     14.92 |
| Attendance               |                91.51 |                     67.9  |
| Sleep_Hours              |                 6.93 |                      7.09 |
| Tutoring_Sessions        |                 1.85 |                      1.15 |
| Physical_Activity        |                 3.07 |                      2.89 |

#### 🏡 COMPARAISON DE L'ENVIRONNEMENT ET CADRE SOCIAL

| Facteur Environnemental        | Top 10% (Réussite)   | Bottom 10% (Difficulté)   |
|:---------

Encore une fois, les tools ont le comportement désiré

# Création du serveur MCP pour le dataset sur les réseaux sociaux et la santé mentale

**Définition du show_mcp_step**

In [52]:
async def show_mcp_steps(agent, question: str):
    """Affiche les etapes d un agent MCP en streaming."""
    print(f"► Question : {question}")
    print("─" * 60)
    async for chunk in agent.astream({"messages": [("user", question)]}):
        if "agent" in chunk:
            msg = chunk["agent"]["messages"][0]
            if hasattr(msg, "tool_calls") and msg.tool_calls:
                for tc in msg.tool_calls:
                    print(f"  ▶ Tool : {tc['name']}({tc['args']})")
            else:
                content = msg.content
                if content:
                    print(f"  ✓ {content[:200]}")
        elif "tools" in chunk:
            obs = chunk["tools"]["messages"][0].content
            print(f"  ◀ Résultat : {str(obs)[:150]}")
    print("─" * 60)
print("✓ show_mcp_steps() prêt")

✓ show_mcp_steps() prêt


### Définition du serveur MCP :

In [62]:
%%writefile mcp_social_health_server.py

from mcp.server.fastmcp import FastMCP
import duckdb, os

mcp = FastMCP('SocialHealthServer')
CSV_PATH = os.path.join(os.path.dirname(__file__), 'Student Social Media And Mental Health Impact.csv')

def get_conn():
    # TODO : créer connexion DuckDB + Student Social Media And Mental Health Impact.csv
    conn = duckdb.connect()
    conn.execute(f"CREATE TABLE IF NOT EXISTS mental_health AS SELECT * FROM read_csv_auto('{CSV_PATH}')")
    return conn

@mcp.tool()
def describe_mental_health() -> str:
    """Retourne une vue générale du dataset sur la santé mentale."""
    # TODO : utiliser get_conn() et retourner un résumé sous forme de string
    conn=get_conn()
    try:
        summary = conn.execute("SUMMARIZE mental_health").df().to_string()
        total_rows = conn.execute("SELECT COUNT(*) FROM mental_health").fetchone()[0]
        return f"Nombre total d'étudiants : {total_rows}\n\nRésumé des colonnes :\n{summary}"
    except Exception as e:
        return f"Erreur lors de la description : {str(e)}"
    finally:
        conn.close()

@mcp.tool()
def query_data(sql: str) -> str:
    """Execute une requête SQL sur le dataset. Table disponible : mental_health.
    Colonnes: Gender, Country, Academic_Level, Most_Used_Platform, Purpose_Of_Use, Stress_Level,
    Age, Avg_Daily_Usage_Hours, Daily_Unlocks, Study_Hours, Physical_Activity_Hours, Sleep_Hours_Per_Night, Mental_Health_Score."""
    conn = get_conn()
    try:
        # Exécution sécurisée/directe de la requête SQL fournie
        result = conn.execute(sql).df().to_string()
        return result
    except Exception as e:
        return f"Erreur lors de l'exécution de la requête : {str(e)}"
    finally:
        conn.close()


@mcp.tool()
def stats_par_segment(colonne_segment: str = 'Country') -> str:
    """
    Filtre et renvoie les moyennes des indicateurs selon un segment précis.
    Valeurs recommandées pour colonne_segment : 'Country', 'Stress_Level', 'Gender', 'Most_Used_Platform'
    """
    if colonne_segment not in ['Country', 'Stress_Level', 'Gender', 'Most_Used_Platform', 'Academic_Level']:
        return "Erreur : Colonne de segmentation non valide."
        
    conn = get_conn()
    try:
        query = f"""
            SELECT {colonne_segment}, 
                   COUNT(*) as Effectif,
                   ROUND(AVG(Age), 1) as Age_Moyen,
                   ROUND(AVG(Avg_Daily_Usage_Hours), 2) as RS_Heures_Moy,
                   ROUND(AVG(Sleep_Hours_Per_Night), 2) as Sommeil_Moy,
                   ROUND(AVG(Mental_Health_Score), 2) as Sante_Mentale_Moy
            FROM mental_health
            GROUP BY {colonne_segment}
            ORDER BY Sante_Mentale_Moy DESC
        """
        return conn.execute(query).df().to_markdown(index=False)
    except Exception as e:
        return f"Erreur : {str(e)}"
    finally:
        conn.close()

@mcp.tool()
def indicateurs_complets_sante_et_social() -> str:
    """
    Renvoie les caractéristiques globales (Moyenne, Min, Max, Médiane) 
    dédiées à la santé mentale et à la consommation des réseaux sociaux.
    """
    conn = get_conn()
    try:
        # Utilisation des fonctions natives DuckDB pour générer un profil complet
        query = """
            SELECT 
                'Mental_Health_Score' as Indicateur, AVG(Mental_Health_Score) as Moyenne, MIN(Mental_Health_Score) as Min, MAX(Mental_Health_Score) as Max, MEDIAN(Mental_Health_Score) as Mediane FROM mental_health
            UNION ALL
            SELECT 
                'Avg_Daily_Usage_Hours' as Indicateur, AVG(Avg_Daily_Usage_Hours) as Moyenne, MIN(Avg_Daily_Usage_Hours) as Min, MAX(Avg_Daily_Usage_Hours) as Max, MEDIAN(Avg_Daily_Usage_Hours) as Mediane FROM mental_health
            UNION ALL
            SELECT 
                'Daily_Unlocks' as Indicateur, AVG(Daily_Unlocks) as Moyenne, MIN(Daily_Unlocks) as Min, MAX(Daily_Unlocks) as Max, MEDIAN(Daily_Unlocks) as Mediane FROM mental_health
            UNION ALL
            SELECT 
                'Sleep_Hours_Per_Night' as Indicateur, AVG(Sleep_Hours_Per_Night) as Moyenne, MIN(Sleep_Hours_Per_Night) as Min, MAX(Sleep_Hours_Per_Night) as Max, MEDIAN(Sleep_Hours_Per_Night) as Mediane FROM mental_health
        """
        res = conn.execute(query).df()
        # Arrondir les résultats pour la propreté du rendu
        for col in ['Moyenne', 'Min', 'Max', 'Mediane']:
            res[col] = res[col].round(2)
            
        return "### 🧠 RESUME ANALYTIQUE DES INDICATEURS CLES\n\n" + res.to_markdown(index=False)
    except Exception as e:
        return f"Erreur : {str(e)}"
    finally:
        conn.close()

if __name__ == '__main__':
    mcp.run()

Overwriting mcp_social_health_server.py


### Code permettant de faire tourner du code asynchrone sur Jupyter avec Windows :

In [63]:
# Hotfix Windows + Jupyter: force errlog=None pour MCP stdio
def patch_mcp_stdio_errlog():
    from contextlib import asynccontextmanager
    import langchain_mcp_adapters.sessions as mcp_sessions
    import mcp.client.stdio as mcp_stdio

    original_stdio_client = mcp_stdio.stdio_client

    @asynccontextmanager
    async def stdio_client_noerr(server, errlog=None):
        async with original_stdio_client(server, errlog=None) as streams:
            yield streams

    mcp_sessions.stdio_client = stdio_client_noerr
    return True

patched = patch_mcp_stdio_errlog()
print(f'✓ Patch MCP stdio errlog applique: {patched}')

✓ Patch MCP stdio errlog applique: True


### Tests du serveur :

**Vérificatio du démarrage du serveur et détection des tools**

In [64]:
# Test — Phase A : vérifier que le serveur démarre et expose les tools
async def test_mental_health_tools():
    client = MultiServerMCPClient({
        'mental_health': {
            'command': sys.executable,
            'args': [os.path.abspath('mcp_social_health_server.py')],
            'transport': 'stdio',
        }
    })
    tools = await client.get_tools()
    print(f'Tools Mental_Health détectés ({len(tools)}) :')
    for t in tools:
        print(f'  ✓ {t.name} — {t.description[:60]}')
    attendus = {'describe_mental_health', 'query_data', 'stats_par_segment', 'indicateurs_complets_sante_et_social'}
    manquants = attendus - {t.name for t in tools}
    if manquants:
        print(f'\n⚠️  Tools manquants : {manquants}')
    else:
        print('\n✅ Les 4 tools sont présents')

await test_mental_health_tools()

# ─────────────────────────────────────────────────────────────────

Tools Mental_Health détectés (4) :
  ✓ describe_mental_health — Retourne une vue générale du dataset sur la santé mentale.
  ✓ query_data — Execute une requête SQL sur le dataset. Table disponible : m
  ✓ stats_par_segment — 
    Filtre et renvoie les moyennes des indicateurs selon un
  ✓ indicateurs_complets_sante_et_social — 
    Renvoie les caractéristiques globales (Moyenne, Min, Ma

✅ Les 4 tools sont présents


**Test des tools du serveur**

In [66]:
# Test  — Phase B : appel réel (à lancer APRÈS implémentation)
# ─────────────────────────────────────────────────────────────────
async def test_mental_health_appel():
     client = MultiServerMCPClient({
         'mental_health': {'command': sys.executable, 'args': [os.path.abspath('mcp_social_health_server.py')], 'transport': 'stdio'}
     })
     tools = await client.get_tools()
     agent = create_react_agent(llm, tools)
     result = await agent.ainvoke({"messages": [("user", "Décris le dataset en une phrase.")]})
     print("Réponse :", result["messages"][-1].content[:300])
await test_mental_health_appel()

Réponse : Le dataset contient des informations sur les étudiants, notamment leur âge, leur sexe, leur pays d'origine, leur niveau académique, la plateforme qu'ils utilisent le plus, l'objectif de leur utilisation, leur durée d'utilisation moyenne, le nombre de fois qu'ils ont ouvert leur appareil, les heures 


### Listing des tools

In [85]:
basic_tools = [describe_dataset, bar_chart, top_values, numeric_stats,make_dashboard]
ia_impact_tools = [analyse_impact_ia_par_groupe,detecter_dependance_ia,impact_temps_utilisation]
perf_tools = [top_progression,analyse_par_type_ecole,facteurs_echec_reussite]

### Fusion des tools

In [86]:
local_tools = basic_tools + ia_impact_tools + perf_tools

### Combinaison de tous les datasets et du serveur MCP

In [73]:
async def run_combined_agent(question : str):
    # 2. Connexion au serveur MCP (Santé Mentale)
    # Remplace par le nom exact de ton fichier serveur validé à l'étape précédente
    mcp_client = MultiServerMCPClient({
        'mental_health': {
            'command': sys.executable,
            'args': [os.path.abspath('mcp_social_health_server.py')],
            'transport': 'stdio',
        }
    })
    
    print("🔄 Connexion au serveur MCP et récupération des outils...")
    # Récupération des outils distants (contenant describe_mental_health, query_data, etc.)
    mcp_tools = await mcp_client.get_tools()
    
    # 3. Fusion de la boîte à outils
    # L'agent verra TOUS ces outils d'un coup et choisira le bon
    all_tools = local_tools + mcp_tools
    print(f"✅ Outils configurés : {len(local_tools)} locaux + {len(mcp_tools)} via MCP.")

    # 4. Initialisation du LLM
    llm = ChatOpenAI(
    model="llama3.2",
    base_url="http://localhost:11434/v1",
    api_key="ollama" ,              # clé fictive requise par l'API
    temperature=0
)

    # 5. Définition des consignes du système (System Prompt)
    SYSTEM_PROMPT = """
        Tu es un expert en data science spécialisé dans l'analyse de la vie étudiante. 
        Tu as accès à 3 datasets interconnectés via tes outils :
        1. L'impact de l'IA sur les étudiants : 'ai_student_impact_dataset.csv'(outils locaux ia_impact_tools et basic_tools)
        2. Les performances scolaires: 'StudentPerformanceFactors.csv' (outils locaux perf_tools et basic_tools)
        3. La santé mentale et les réseaux sociaux (via le serveur MCP mental_health)
        Règles cruciales :
        - Utilise TOUJOURS le format Markdown pour tes réponses.
        - Analyse les données objectivement en croisant les résultats si l'utilisateur te le demande.
        - Si l'outil MCP 'query_data' est requis pour une question complexe sur la santé mentale, 
        génère la requête SQL appropriée.
    """

    # 6. Création de l'agent ReAct (Raisonnement + Action)
    print(f"🤖 Initialisation de l'agent avec la question : '{question}'\n")
    agent = create_react_agent(llm, tools=all_tools, prompt=SYSTEM_PROMPT)

    result = await agent.ainvoke({'messages': [('user', question)]})



    # 7. Affichage détaillé des étapes (Show Steps)
    messages = result.get('messages', [])
    print(f'\n' + '='*60)
    print(f'Nombre de messages : {len(messages)}')
    
    for i, msg in enumerate(messages):
        t = type(msg).__name__
        
        # Affichage du contenu textuel (Pensées de l'IA, réponses ou retours d'outils)
        if hasattr(msg, 'content') and msg.content:
            content = str(msg.content)[:200].replace('\n', ' ') # Nettoie les retours à la ligne pour l'affichage
            print(f'  [{i}] {t}: {content}...')
            
        # Affichage si l'agent a décidé de déclencher un outil
        if hasattr(msg, 'tool_calls') and msg.tool_calls:
            for tc in msg.tool_calls:
                print(f'  [{i}] → Tool call: {tc["name"]}({tc.get("args", {})})')
                
    print('='*60 + '\n')
    
    # 8. Restitution de la réponse finale
    final = messages[-1].content if messages else 'Aucune réponse'
    print(f'Réponse finale :\n{final}')
    


### Test de l'agent complet (sans mémoire)

In [77]:
# --- EXEMPLES DE TESTS ---
    
# Test 1 : Question sur l'IA (Outil Local)
print("\n--- Test 1 : Question IA (Local) ---")
query_1 = "Quel est l'impact du temps d'utilisation de l'IA sur l'évolution du GPA des étudiants ?"
await run_combined_agent(query_1)




--- Test 1 : Question IA (Local) ---
🔄 Connexion au serveur MCP et récupération des outils...
✅ Outils configurés : 11 locaux + 4 via MCP.
🤖 Initialisation de l'agent avec la question : 'Quel est l'impact du temps d'utilisation de l'IA sur l'évolution du GPA des étudiants ?'


Nombre de messages : 4
  [0] HumanMessage: Quel est l'impact du temps d'utilisation de l'IA sur l'évolution du GPA des étudiants ?...
  [1] → Tool call: impact_temps_utilisation({'filepath': 'ai_student_impact_dataset.csv'})
  [2] ToolMessage: ### ⏱️ ANALYSE DE L'IMPACT DU TEMPS D'UTILISATION DE L'IA  Ce tableau compare le profil des étudiants selon leur volume horaire hebdomadaire sur les outils d'IA :  | Tranche_Utilisation          |   E...
  [3] AIMessage: ### Résultats de l'analyse  L'impact du temps d'utilisation de l'IA sur l'évolution du GPA des étudiants est complexe et peut varier en fonction de la tranche d'utilisation.  *   Les étudiants qui uti...

Réponse finale :
### Résultats de l'analyse

L'impa

In [75]:

    # Test 2 : Question sur la Santé Mentale (Outil MCP)
print("\n--- Test 2 : Question Santé Mentale (MCP) ---")
query_2 = "Donne-moi le résumé analytique des indicateurs de santé mentale et dis-moi s'il y a des extrêmes."
await run_combined_agent(query_2)
            


--- Test 2 : Question Santé Mentale (MCP) ---
🔄 Connexion au serveur MCP et récupération des outils...
✅ Outils configurés : 11 locaux + 4 via MCP.
🤖 Initialisation de l'agent avec la question : 'Donne-moi le résumé analytique des indicateurs de santé mentale et dis-moi s'il y a des extrêmes.'


Nombre de messages : 4
  [0] HumanMessage: Donne-moi le résumé analytique des indicateurs de santé mentale et dis-moi s'il y a des extrêmes....
  [1] → Tool call: describe_mental_health({})
  [2] ToolMessage: [{'type': 'text', 'text': "Nombre total d'étudiants : 5000\n\nRésumé des colonnes :\n                column_name column_type          min            max  approx_unique                 avg             ...
  [3] AIMessage: **Résumé des indicateurs de santé mentale**  Les données fournies contiennent des informations sur la santé mentale des étudiants, notamment leur niveau de stress, leur score de santé mentale et leurs...

Réponse finale :
**Résumé des indicateurs de santé mentale**

Les d

In [76]:
# Test 3 : Requête SQL générée par l'agent (Outil MCP query_data)
print("\n--- Test 3 : Requête Complexe (MCP query_data) ---")
query_3 = "Fais-moi un classement des 3 pays où le score moyen de santé mentale est le plus bas."
await run_combined_agent(query_3)


--- Test 3 : Requête Complexe (MCP query_data) ---
🔄 Connexion au serveur MCP et récupération des outils...
✅ Outils configurés : 11 locaux + 4 via MCP.
🤖 Initialisation de l'agent avec la question : 'Fais-moi un classement des 3 pays où le score moyen de santé mentale est le plus bas.'


Nombre de messages : 4
  [0] HumanMessage: Fais-moi un classement des 3 pays où le score moyen de santé mentale est le plus bas....
  [1] → Tool call: query_data({'sql': 'SELECT AVG(Mental_Health_Score) FROM mental_health GROUP BY Country ORDER BY AVG(Mental_Health_Score) DESC LIMIT 3'})
  [2] ToolMessage: [{'type': 'text', 'text': '   avg(Mental_Health_Score)\n0                     8.350\n1                     8.100\n2                     8.075', 'id': 'lc_3e70760a-d59e-4746-a74e-74e7fa6090fd'}]...
  [3] AIMessage: **Classement des pays avec le score moyen de santé mentale le plus bas**  Selon les données disponibles, voici le classement des 3 pays où le score moyen de santé mentale est le plus bas 

L'agent fonctionne comme on le désire.

# Création d'une interface utilisateur :

**Comme on créer un code app.py, il faut redéfinir chaque fonctions et donc regrouper tout le code précédent.**

**Cet agent bénéficie également d'un système de mémoire**

In [190]:
%%writefile app.py
# Étape 6 — Votre application Streamlit complète
import re, glob
import streamlit as st
import plotly.express as px
import asyncio, os, sys, json, re
import pandas as pd, plotly.graph_objects as go
from plotly.subplots import make_subplots
import nest_asyncio; nest_asyncio.apply()
import webbrowser

os.chdir(os.path.dirname(os.path.abspath(__file__)))

from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain_core.tools import tool




#-----Redéfinition des fonctions-----
def fuzzy_col(df, col_name: str) -> str:
    """Trouve la vraie colonne dans df à partir d'un nom approximatif."""
    col_clean = re.sub(r'[^a-zA-Z0-9 ]', '', col_name).strip().lower()
    for c in df.columns:
        if c.lower() == col_clean:
            return c
    for c in df.columns:
        if col_clean in c.lower() or c.lower() in col_clean:
            return c
    words = [w for w in col_clean.split() if len(w) >= 4]
    for word in words:
        stems = [word,
                 word[:-1] if len(word) > 4 else '',
                 word[:-2] if len(word) > 5 else '']
        for c in df.columns:
            if any(s and s in c.lower() for s in stems):
                return c
    return col_name

def _col_not_found_msg(df, col_name, kind='catégorielle'):
    if kind == 'catégorielle':
        available = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    else:
        available = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return (f"Colonne '{col_name}' introuvable. "
            f"Utilise un de ces noms exacts : {available}")

def _clean_filepath(filepath: str) -> str:
    """Sanitise le filepath : JSON array, faux chemin absolu, typo de nom."""
    fp = filepath.strip()
    # Cas 1 : tableau JSON  ["file.csv"]
    if fp.startswith("["):
        fp = fp.strip("[]").split(",")[0].strip()
        fp = fp.strip('"').strip("'")
    # Cas 2 : faux chemin absolu inventé par le LLM (/path/to/file.csv)
    if not os.path.exists(fp) and "/" in fp:
        basename = fp.split("/")[-1]
        if os.path.exists(basename):
            fp = basename
    # Cas 3 : typo dans le nom (job_posting.csv → job_postings.csv)
    if not os.path.exists(fp):
        name = os.path.basename(fp)
        candidates = glob.glob("*.csv")
        # prefer exact prefix match
        for c in candidates:
            if c.startswith(name.split(".")[0][:8]):
                fp = c
                break
        else:
            if len(candidates) == 1:
                fp = candidates[0]
    return fp




    
#-----Redéfinition des tools-----
#-----Tools Basics------
@tool
def describe_dataset(filepath: str = "ai_student_impact_dataset.csv") -> str:
    """Charge un CSV et retourne ses dimensions et colonnes. Utiliser EN PREMIER.
    Args:
        filepath: Chemin vers le fichier CSV à analyser.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'ai_student_impact_dataset.csv'."
    cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
    num_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    return (f"{df.shape[0]} lignes, {df.shape[1]} colonnes.\n"
            f"Catégorielles (utilise bar_chart, noms exacts) : {cat_cols}\n"
            f"Numériques (utilise numeric_stats, noms exacts) : {num_cols}")

@tool
def bar_chart(filepath: str = "ai_student_impact_dataset.csv", column: str = "", title: str = '') -> str:
    """Génère un graphique à barres horizontal des 10 valeurs les plus fréquentes d'une colonne.
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne catégorielle.
        title: Titre du graphique (optionnel).
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable."
        
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
        
    top = df[col].value_counts().head(10).reset_index()
    top.columns = [col, 'count']
    
    # Création du graphique Plotly
    fig = px.bar(top.sort_values('count'), x='count', y=col, orientation='h', title=title or f"Top 10 — {col}")
    
    # Stockage de la figure dans le session_state pour que Streamlit puisse l'afficher
    st.session_state["last_generated_chart"] = fig
    
    return f"✅ Le graphique pour la colonne '{col}' a été généré avec succès et est prêt à être affiché à l'écran."

@tool
def scatter_correlation(filepath: str, x_col: str, y_col: str, color_col: str = None,title: str = '') -> str:
    """Génère un graphique de corrélation (nuage de points) entre deux colonnes numériques. Sauvegarde en HTML.
    Args:
        filepath: Chemin vers le fichier CSV.
        x_col: Nom de la colonne numérique pour l'axe X.
        y_col: Nom de la colonne numérique pour l'axe Y.
        color_col: Colonne optionnelle pour colorer les points (catégorielle).
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable."

    x = fuzzy_col(df, x_col)
    y = fuzzy_col(df, y_col)
    c = fuzzy_col(df, color_col) if color_col else None
    
    if x not in df.columns or y not in df.columns:
        return f"Colonnes introuvables pour le scatter plot. X: '{x}', Y: '{y}'."

    fig = px.scatter(df, x=x, y=y, color=c, title=f"Corrélation : {x} vs {y}")
    
    # Sécuriser le nom du fichier HTML de sortie
    filename_x = x.replace(" ", "_").replace("/", "_")
    filename_y = y.replace(" ", "_").replace("/", "_")
    out = f'scatter_{filename_x}_vs_{filename_y}.html'
    
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Graphique de corrélation sauvegardé et ouvert : {out}"


@tool
def pie_distribution(filepath: str, column: str, title: str ='') -> str:
    """Génère un graphique en camembert (Pie chart) pour analyser la répartition (top 5). Sauvegarde en HTML.
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom de la colonne catégorielle à analyser.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable."
        
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
    
    top = df[col].value_counts().head(5).reset_index()
    top.columns = [col, 'count']
    
    fig = px.pie(top, values='count', names=col, title=f"Répartition (Top 5) de {col}")
    
    # Sécuriser le nom du fichier HTML de sortie
    out = f'pie_{col.replace(" ", "_").replace("/", "_")}.html'
    
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Graphique en camembert sauvegardé et ouvert : {out}"


@tool
def top_values(filepath: str = "ai_student_impact_dataset.csv", column: str = "") -> str:
    """Retourne les 10 valeurs (s'il y en a 10) les plus fréquentes d'une colonne catégorielle (texte).
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne catégorielle.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'ai_student_impact_dataset.csv'."
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'catégorielle')
    result = df[col].value_counts()
    n_items = len(result)
    if n_items>10:
        result=result.head(10)
        titre = f"Top 10 — {col} :\n"
    else:
        titre = f"Top {n_items} — {col} :\n"
    return titre + result.to_string()



@tool
def numeric_stats(filepath: str, column: str) -> str:
    """TODO : Retourne min, max, moyenne, médiane d'une colonne numérique
    Args:
        filepath: Chemin vers le fichier CSV.
        column: Nom exact ou partiel de la colonne numérique.
    """
    try:
        df = pd.read_csv(_clean_filepath(filepath))
    except FileNotFoundError:
        return f"Fichier '{filepath}' introuvable. Utilise 'ai_student_impact_dataset.csv'."
    col = fuzzy_col(df, column)
    if col not in df.columns:
        return _col_not_found_msg(df, column, 'numérique')
    serie = df[col].dropna()

    return (
    f"Statistiques — {col} :\n"
    f"Moyenne : {serie.mean():.2f}\n"
    f"Médiane : {serie.median():.2f}\n"
    f"Minimum : {serie.min():.2f}\n"
    f"Maximum : {serie.max():.2f}"
)
    pass


@tool
def make_dashboard(filepath: str, column1: str, column2: str, title: str) -> str:
    """Génère un dashboard HTML avec 2 bar charts côte à côte. Sauvegarde et ouvre dans le navigateur.
    IMPORTANT : utilise uniquement des noms de colonnes retournés par describe_dataset.
    Args:
        filepath: Chemin vers le fichier CSV.
        column1: Première colonne catégorielle (nom exact ou partiel).
        column2: Deuxième colonne catégorielle (nom exact ou partiel).
        title: Titre du dashboard.
    """
    df = pd.read_csv(_clean_filepath(filepath))

    # Normaliser et vérifier les deux colonnes
    col1 = fuzzy_col(df, column1)
    col2 = fuzzy_col(df, column2)

    # Si une colonne est introuvable → retourner la liste des colonnes disponibles
    # (le LLM lira ce message et relancera avec les bons noms)
    if col1 not in df.columns or col2 not in df.columns:
        cat_cols = [c for c in df.columns if not pd.api.types.is_numeric_dtype(df[c])]
        missing = [c for c, real in [(column1, col1), (column2, col2)] if real not in df.columns]
        return (f"Colonnes introuvables : {missing}. "
                f"Utilise ces noms exacts : {cat_cols}")

    # Préparer les données : top 10 pour chaque colonne
    def top10(col):
        t = df[col].value_counts().head(10).reset_index()
        t.columns = [col, 'count']
        return t.sort_values('count')

    # Créer la figure avec 2 sous-graphiques côte à côte
    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=[col1, col2],
                        horizontal_spacing=0.15)
    t1 = top10(col1)
    fig.add_trace(go.Bar(x=t1['count'], y=t1[col1], orientation='h', name=col1), row=1, col=1)
    t2 = top10(col2)
    fig.add_trace(go.Bar(x=t2['count'], y=t2[col2], orientation='h', name=col2), row=1, col=2)
    fig.update_layout(title_text=title, height=500, showlegend=False)

    out = 'dashboard.html'
    fig.write_html(out)
    webbrowser.open(f'file://{os.path.abspath(out)}')
    return f"Dashboard sauvegardé : {out} ({col1} + {col2})"

#------Tools Impact IA-----
@tool
def analyse_impact_ia_par_groupe(filepath: str,column: str = 'Primary_Use_Case') -> str:
    """
    Analyse l'impact de l'IA en regroupant par une colonne catégorielle spécifique.
    Colonnes valides recommandées : 'Primary_Use_Case', 'Prompt_Engineering_Skill', 'Major_Category'
    """
    colonnes_valides = ['Primary_Use_Case', 'Prompt_Engineering_Skill', 'Major_Category', 'Burnout_Risk_Level']
    if column not in colonnes_valides:
        return f"Erreur : La colonne de segmentation doit être l'une des suivantes : {colonnes_valides}"
    
    df_ia = pd.read_csv(_clean_filepath(filepath))
        
    # Groupement et calcul des moyennes clés
    analyse = df_ia.groupby(column).agg({
        'Pre_Semester_GPA': 'mean',
        'Post_Semester_GPA': 'mean',
        'Weekly_GenAI_Hours': 'mean',
        'Skill_Retention_Score': 'mean',
        'Anxiety_Level_During_Exams': 'mean'
    }).round(2).reset_index()
    
    # Ajout d'une colonne calculée pour voir l'évolution globale du GPA par groupe
    analyse["Evolution_GPA"] = (analyse["Post_Semester_GPA"] - analyse["Pre_Semester_GPA"]).round(2)
    
    rapport = f"### 🤖 ANALYSE DE L'IA SEGMENTÉE PAR : {column}\n\n"
    rapport += analyse.to_markdown(index=False)
    
    return rapport

@tool
def detecter_dependance_ia(filepath: str,seuil_heures: int = 15) -> str:
    """
    Filtre les étudiants ayant une utilisation intensive de l'IA (défini par le seuil d'heures)
    et compare leur profil à la moyenne globale de tous les étudiants.
    """
    df_ia = pd.read_csv(_clean_filepath(filepath))

    # Groupe "Intensif" vs Groupe "Global"
    df_intensif = df_ia[df_ia['Weekly_GenAI_Hours'] >= seuil_heures]
    
    metrics = [
        'Pre_Semester_GPA', 'Post_Semester_GPA', 'Weekly_GenAI_Hours', 
        'Traditional_Study_Hours', 'Anxiety_Level_During_Exams', 'Skill_Retention_Score'
    ]
    
    moyenne_intensif = df_intensif[metrics].mean().to_frame(name=f'Utilisateurs Intensifs (>= {seuil_heures}h)')
    moyenne_globale = df_ia[metrics].mean().to_frame(name='Moyenne Globale')
    
    # Fusion des deux analyses pour comparaison directe
    comparatif = moyenne_globale.join(moyenne_intensif).round(2).reset_index()
    comparatif.rename(columns={'index': 'Indicateurs'}, inplace=True)
    
    rapport = f"### ⚠️ PROFIL COMPARAISON : FORTE DÉPENDANCE À L'IA (Effectif : {len(df_intensif)} étudiants)\n\n"
    rapport += comparatif.to_markdown(index=False)
    
    return rapport


@tool
def impact_temps_utilisation(filepath: str) -> str:
    """
    Analyse l'impact du temps d'utilisation hebdomadaire de l'IA sur le succès académique 
    (GPA, rétention des compétences) et le bien-être (anxiété, burnout) des étudiants.
    """
    df_ia = pd.read_csv(_clean_filepath(filepath))
    df = df_ia.copy()
    
    # 1. Définition des tranches d'utilisation basées sur 'Weekly_GenAI_Hours'
    # Moins de 5h = Faible | Entre 5h et 15h = Modéré | Plus de 15h = Intensif
    def segmenter_heures(heures):
        if heures < 5:
            return "1. Faible (< 5h/semaine)"
        elif heures <= 15:
            return "2. Modéré (5h - 15h/semaine)"
        else:
            return "3. Intensif (> 15h/semaine)"
            
    df['Tranche_Utilisation'] = df['Weekly_GenAI_Hours'].apply(segmenter_heures)
    
    # 2. Agrégation des statistiques clés par tranche d'utilisation
    analyse_impact = df.groupby('Tranche_Utilisation').agg({
        'Student_ID': 'count',                 # Nombre d'étudiants dans le groupe
        'Pre_Semester_GPA': 'mean',
        'Post_Semester_GPA': 'mean',
        'Traditional_Study_Hours': 'mean',     # Permet de voir s'ils étudient moins "traditionnellement"
        'Skill_Retention_Score': 'mean',       # Impact sur l'apprentissage profond
        'Anxiety_Level_During_Exams': 'mean'   # Impact psychologique
    }).round(2).reset_index()
    
    # Renommer la colonne count pour plus de clarté
    analyse_impact.rename(columns={'Student_ID': 'Effectif_Etudiants'}, inplace=True)
    
    # 3. Calcul de l'évolution du GPA pour chaque groupe
    analyse_impact["Evolution_GPA"] = (analyse_impact["Post_Semester_GPA"] - analyse_impact["Pre_Semester_GPA"]).round(2)
    
    # Réorganisation des colonnes pour une lecture logique par l'agent
    colonnes_ordre = [
        'Tranche_Utilisation', 'Effectif_Etudiants', 'Traditional_Study_Hours',
        'Pre_Semester_GPA', 'Post_Semester_GPA', 'Evolution_GPA',
        'Skill_Retention_Score', 'Anxiety_Level_During_Exams'
    ]
    
    # Construction du rapport Markdown
    rapport = "### ⏱️ ANALYSE DE L'IMPACT DU TEMPS D'UTILISATION DE L'IA\n\n"
    rapport += "Ce tableau compare le profil des étudiants selon leur volume horaire hebdomadaire sur les outils d'IA :\n\n"
    rapport += analyse_impact[colonnes_ordre].to_markdown(index=False)
    
    return rapport


    
#------Tools Performances-----
@tool
def top_progression(filepath : str,top_n: int = 15) -> str:
    """
    Calcule la progression des étudiants (Exam_Score - Previous_Scores),
    génère le top N des meilleures progressions et fournit la moyenne de leur profil.
    """

    df_perf = pd.read_csv(_clean_filepath(filepath))
    df = df_perf.copy()
    
    # Calcul de la variable de progression
    df['Progression'] = df['Exam_Score'] - df['Previous_Scores']
    
    # Extraction du Top N
    top_students = df.sort_values(by='Progression', ascending=False).head(top_n)
    
    # Sélection des colonnes numériques clés pour l'analyse
    cols_numeriques = ['Progression', 'Hours_Studied', 'Attendance', 'Sleep_Hours', 'Tutoring_Sessions']
    cols_cat = ['Motivation_Level', 'Teacher_Quality', 'Parental_Involvement']
    
    # 1. Calcul des moyennes du groupe d'élite
    moyennes = top_students[cols_numeriques].mean().to_frame().T
    moyennes.index = ['MOYENNE DU TOP']
    
    # 2. Préparation du tableau des étudiants
    affichage_students = top_students[cols_numeriques + cols_cat].copy()
    
    # Construction du rendu textuel final pour l'agent
    rapport = f"### 📈 TOP {top_n} DES MEILLEURES PROGRESSIONS APPRÉCIABLES\n\n"
    rapport += affichage_students.to_markdown(index=False)
    rapport += "\n\n### 📊 PROFIL STATISTIQUE MOYEN DE CE GROUPE\n\n"
    rapport += moyennes.to_markdown(index=False)
    
    return rapport

@tool
def analyse_par_type_ecole(filepath : str) -> str:
    """
    Compare les performances et l'accès aux ressources entre les écoles publiques et privées.
    """

    df_perf = pd.read_csv(_clean_filepath(filepath))

    # Agrégation des notes et variables critiques par School_Type
    stats_ecole = df_perf.groupby('School_Type').agg({
        'Exam_Score': 'mean',
        'Attendance': 'mean',
        'Hours_Studied': 'mean',
        'Previous_Scores': 'mean'
    }).round(2).reset_index()
    
    # Distribution des ressources en % (ex: Internet_Access)
    distribution_internet = pd.crosstab(df_perf['School_Type'], df_perf['Internet_Access'], normalize='index') * 100
    distribution_internet = distribution_internet.round(1).reset_index()
    
    rapport = "### 🏢 COMPARAISON PUBLIC VS PRIVÉ (MOYENNES)\n\n"
    rapport += stats_ecole.to_markdown(index=False)
    rapport += "\n\n### 🌐 ACCÈS À INTERNET PAR TYPE D'ÉCOLE (EN %)\n\n"
    rapport += distribution_internet.to_markdown(index=False)
    
    return rapport

@tool
def facteurs_echec_reussite(filepath : str) -> str:
    """
    Isole les 10% des meilleurs étudiants et les 10% des étudiants en difficulté 
    sur leur Exam_Score, puis compare leurs profils (habitudes d'étude, sommeil, environnement).
    """
    df_perf = pd.read_csv(_clean_filepath(filepath))
    df = df_perf.copy()
    
    # 1. Détermination des seuils pour les top 10% et bottom 10%
    seuil_haut = df['Exam_Score'].quantile(0.90)
    seuil_bas = df['Exam_Score'].quantile(0.10)
    
    # 2. Création des deux sous-groupes
    df_top = df[df['Exam_Score'] >= seuil_haut]
    df_bottom = df[df['Exam_Score'] <= seuil_bas]
    
    # 3. Métriques numériques clés à comparer
    metrics_num = [
        'Exam_Score', 'Hours_Studied', 'Attendance', 
        'Sleep_Hours', 'Tutoring_Sessions', 'Physical_Activity'
    ]
    
    moyennes_top = df_top[metrics_num].mean().to_frame(name='Top 10% (Réussite)')
    moyennes_bottom = df_bottom[metrics_num].mean().to_frame(name='Bottom 10% (Difficulté)')
    
    # Fusion des analyses numériques
    comparatif_num = moyennes_top.join(moyennes_bottom).round(2).reset_index()
    comparatif_num.rename(columns={'index': 'Indicateurs Numériques'}, inplace=True)
    
    # 4. Métriques catégorielles clés (Facteurs environnementaux)
    # On va calculer le taux d'accès ou de forte implication pour donner du contexte social
    facteurs_env = []
    
    # % Accès Internet Élevé
    facteurs_env.append({
        'Facteur Environnemental': 'Accès Internet (Oui %)',
        'Top 10% (Réussite)': f"{(df_top['Internet_Access'] == 'Yes').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Internet_Access'] == 'Yes').mean() * 100:.1f}%"
    })
    
    # % Implication Parentale Forte
    facteurs_env.append({
        'Facteur Environnemental': 'Implication Parentale (High %)',
        'Top 10% (Réussite)': f"{(df_top['Parental_Involvement'] == 'High').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Parental_Involvement'] == 'High').mean() * 100:.1f}%"
    })
    
    # % Motivation Élevée
    facteurs_env.append({
        'Facteur Environnemental': 'Motivation de l\'élève (High %)',
        'Top 10% (Réussite)': f"{(df_top['Motivation_Level'] == 'High').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Motivation_Level'] == 'High').mean() * 100:.1f}%"
    })
    
    # % Qualité Enseignement Excellente
    facteurs_env.append({
        'Facteur Environnemental': 'Qualité Enseignant (High %)',
        'Top 10% (Réussite)': f"{(df_top['Teacher_Quality'] == 'High').mean() * 100:.1f}%",
        'Bottom 10% (Difficulté)': f"{(df_bottom['Teacher_Quality'] == 'High').mean() * 100:.1f}%"
    })
    
    comparatif_cat = pd.DataFrame(facteurs_env)
    
    # 5. Construction du rapport final
    rapport = f"### 🔑 ANALYSE DES FACTEURS DE RÉUSSITE VS ÉCHEC\n"
    rapport += f"Échantillon basé sur les extrêmes du jeu de données (Seuil Réussite >= {seuil_haut} pts | Seuil Difficulté <= {seuil_bas} pts).\n\n"
    
    rapport += "#### 📊 COMPARAISON DES HABITUDES (MOYENNES)\n\n"
    rapport += comparatif_num.to_markdown(index=False)
    
    rapport += "\n\n#### 🏡 COMPARAISON DE L'ENVIRONNEMENT ET CADRE SOCIAL\n\n"
    rapport += comparatif_cat.to_markdown(index=False)
    
    return rapport




st.set_page_config(page_title="Agent Student", page_icon="🏙️", layout="wide")
st.title("🏙️ Agent Student")

llm = ChatOpenAI(model="llama3.2", base_url="http://localhost:11434/v1", api_key="ollama", temperature=0)

basic_tools = [describe_dataset, bar_chart, top_values, numeric_stats,make_dashboard,pie_distribution,scatter_correlation]
ia_impact_tools = [analyse_impact_ia_par_groupe,detecter_dependance_ia,impact_temps_utilisation]
perf_tools = [top_progression,analyse_par_type_ecole,facteurs_echec_reussite]

local_tools = basic_tools + ia_impact_tools + perf_tools
local_tools_dict = {tool.name: tool for tool in local_tools}

# Sauvegarder l'instance de mémoire dans la session de Streamlit pour éviter qu'elle soit recréée
if "agent_memory" not in st.session_state:
    st.session_state["agent_memory"] = MemorySaver()

# On utilise cette instance persistante
memory = st.session_state["agent_memory"]


def build_home_dashboard() -> go.Figure:
    """Génère un dashboard d'accueil 2x2 combinant les trois thématiques du projet."""
    fig = make_subplots(
        rows=2, cols=2,
        subplot_titles=(
            "Répartition des Étudiants par Filière (Major)", 
            "Top Utilisation Principale de l'IA", 
            "Impact des Heures d'Étude sur la Note d'Examen", 
            "Santé mentale en fonction de l'utilisation des réseaux sociaux"
        ),
        specs=[
            [{"type": "pie"}, {"type": "bar"}],
            [{"type": "scatter"}, {"type": "box"}]
        ],
        vertical_spacing=0.15,
        horizontal_spacing=0.10
    )

    # ---- FIG 1 : Pie Chart - Répartition par Major Category (Dataset IA) ----
    try:
        df_ia = pd.read_csv("ai_student_impact_dataset.csv")
        df_major = df_ia['Major_Category'].value_counts().reset_index()
        df_major.columns = ['Major_Category', 'count']
        fig_pie = px.pie(df_major, values='count', names='Major_Category', color_discrete_sequence=px.colors.qualitative.Pastel)
        for trace in fig_pie.data:
            fig.add_trace(trace, row=1, col=1)
    except Exception:
        pass

    # ---- FIG 2 : Bar Chart - Primary Use Case de l'IA (Dataset IA) ----
    try:
        df_use = df_ia['Primary_Use_Case'].value_counts().head(5).reset_index()
        df_use.columns = ['Primary_Use_Case', 'count']
        fig_bar = px.bar(df_use, x='count', y='Primary_Use_Case', orientation='h', color='count', color_continuous_scale='Blues')
        for trace in fig_bar.data:
            fig.add_trace(trace, row=1, col=2)
    except Exception:
        pass

    # ---- FIG 3 : Scatter Plot - Heures d'étude vs Exam Score (Dataset Performance) ----
    try:
        df_perf = pd.read_csv("StudentPerformanceFactors.csv")
        # Échantillon de 500 points max pour éviter de surcharger le graphique
        df_perf_sample = df_perf.sample(n=min(500, len(df_perf)), random_state=42)
        fig_scat = px.scatter(df_perf_sample, x='Hours_Studied', y='Exam_Score', opacity=0.6, color='Motivation_Level')
        for trace in fig_scat.data:
            fig.add_trace(trace, row=2, col=1)
    except Exception:
        pass

   # ---- FIG 4 : Scatter Plot - Score de Santé Mentale vs Temps d'utilisation quotidien ----
    try:
        df_mental = pd.read_csv("Student Social Media And Mental Health Impact.csv")
        df_mental_sample = df_mental.sample(n=min(500, len(df_mental)), random_state=42)
        
        fig_scat = px.scatter(df_mental_sample, x='Avg_Daily_Usage_Hours', y='Mental_Health_Score', opacity=0.6, color='Sleep_Hours_Per_Night')
        for trace in fig_scat.data:
            fig.add_trace(trace, row=2, col=2)
    except Exception:
        pass

    # ---- Ligne 2, Col 1 : Heures d'étude vs Exam Score ----
    fig.update_xaxes(title_text="Heures étudiées", row=2, col=1)
    fig.update_yaxes(title_text="Note à l'examen", row=2, col=1)

    # ---- Ligne 2, Col 2 : Santé Mentale vs Réseaux Sociaux ----
    fig.update_xaxes(title_text="Utilisation quotidienne (Heures)", row=2, col=2)
    fig.update_yaxes(title_text="Score de santé mentale", row=2, col=2)

    # Ajustements globaux du layout
    fig.update_layout(
        height=800,
        showlegend=False,
        template="plotly_white",
        title_text="📊 Tableau de Bord Global des Indicateurs Étudiants",
        title_font_size=20,
        title_x=0.5,
        margin=dict(l=30, r=30, t=80, b=40)
    )
    
    return fig

dashboard_fig = build_home_dashboard()
st.plotly_chart(dashboard_fig, use_container_width=True)






st.markdown("---")
async def run_combined_agent(question : str):
    # 2. Connexion au serveur MCP (Santé Mentale)
    # Remplace par le nom exact de ton fichier serveur validé à l'étape précédente
    mcp_client = MultiServerMCPClient({
        'mental_health': {
            'command': sys.executable,
            'args': [os.path.abspath('mcp_social_health_server.py')],
            'transport': 'stdio',
        }
    })

    

    config = {"configurable": {"thread_id": "thread_student"}}
    
    print("🔄 Connexion au serveur MCP et récupération des outils...")
    # Récupération des outils distants (contenant describe_mental_health, query_data, etc.)
    mcp_tools = await mcp_client.get_tools()
    
    # 3. Fusion de la boîte à outils
    # L'agent verra TOUS ces outils d'un coup et choisira le bon
    all_tools = local_tools + mcp_tools
    print(f"✅ Outils configurés : {len(local_tools)} locaux + {len(mcp_tools)} via MCP.")

    # 5. Définition des consignes du système (System Prompt)
    SYSTEM_PROMPT = """
        Tu es un expert en data science spécialisé dans l'analyse de la vie étudiante. 
        Tu as accès à 3 datasets interconnectés via tes outils :
        1. L'impact de l'IA sur les étudiants : 'ai_student_impact_dataset.csv'(outils locaux ia_impact_tools et basic_tools)
        2. Les performances scolaires: 'StudentPerformanceFactors.csv' (outils locaux perf_tools et basic_tools)
        3. La santé mentale et les réseaux sociaux (via le serveur MCP mental_health)
        Règles cruciales :
        - Utilise TOUJOURS le format Markdown pour tes réponses.
        - Analyse les données objectivement en croisant les résultats si l'utilisateur te le demande.
        - Si l'outil MCP 'query_data' est requis pour une question complexe sur la santé mentale, 
        génère la requête SQL appropriée.
        - Quand on te demande de comparer des données ou des chiffres issus d'un outil précédent, retourne les valeurs exactes et ne fais pas de généralisations
    """

    # 6. Création de l'agent ReAct (Raisonnement + Action)
    print(f"🤖 Initialisation de l'agent avec la question : '{question}'\n")
    agent = create_react_agent(llm, tools=all_tools, prompt=SYSTEM_PROMPT,checkpointer=memory)

    


    steps = []
    
    # 1. Une SEULE exécution complète et stable
    result = await agent.ainvoke({"messages": [("user", question)]},config)
    
    # 2. On extrait les outils utilisés pour alimenter l'expander Streamlit
    messages = result.get("messages", [])
    for msg in messages:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            for tool_call in msg.tool_calls:
                steps.append(f"⚙️ **Appel de l'outil :** `{tool_call['name']}` avec `{tool_call['args']}`")
        elif msg.type == "tool":
            steps.append(f"👁️ **Résultat obtenu** de l'outil `{msg.name}`.")
            
    # 3. Récupération du message final écrit par le LLM
    answer = messages[-1].content
    return answer, steps
st.subheader("🤖 Posez vos questions à l'Assistant IA")
st.caption("Exemple : 'Quel est l'impact du temps d'utilisation de l'IA sur l'évolution du GPA des étudiants ?'")

# Initialisation de la mémoire du chat Streamlit
if "messages" not in st.session_state:
    st.session_state.messages = []

# Affichage des messages historiques
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# Capture d'une nouvelle entrée utilisateur
if user_query := st.chat_input("Votre message..."):
    # Affichage du message utilisateur
    with st.chat_message("user"):
        st.markdown(user_query)
    st.session_state.messages.append({"role": "user", "content": user_query})
    
    # Génération de la réponse de l'IA
    with st.chat_message("assistant"):
        with st.spinner("L'agent réfléchit et consulte le serveur MCP..."):
            try:
                # Exécution de la boucle asynchrone requise par le client MCP
                loop = asyncio.get_event_loop()
                answer, steps = loop.run_until_complete(run_combined_agent(user_query))
                
                # Affichage des étapes de raisonnement dans un menu accordéon déroulant
                if steps:
                    with st.expander("🔍 Voir le cheminement et les outils utilisés"):
                        for step in steps:
                            st.write(step)

                

                
                # Affichage de la réponse finale
                st.markdown(answer)
                st.session_state.messages.append({"role": "assistant", "content": answer})
                
            except Exception as e:
                error_msg = f"❌ Une erreur est survenue lors de l'appel de l'agent : {str(e)}"
                st.error(error_msg)
    



Overwriting app.py


### Lancement de l'application :

In [191]:
# Lancez l'application Streamlit
import subprocess
subprocess.Popen([sys.executable, '-m', 'streamlit', 'run', 'app.py'])
print("✓ Streamlit lancé → http://localhost:8501")

✓ Streamlit lancé → http://localhost:8501


L'agent fonctionne même s'il a des petits problèmes en ce qui concerne la compréhension des requêtes utilisateurs.

Il fournit un dashboard au démarrage et son système de mémoire est opérationnel.